# Ambient Email Agent — walkthrough notebook
Runs every milestone step by step. Works offline (FakeLLM) or with Gemini if `GOOGLE_API_KEY` is in `.env`.

Select the `.venv` kernel, then **Run All**.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))   # run from the project root
from dotenv import load_dotenv; load_dotenv()
from langgraph.checkpoint.memory import InMemorySaver
from agent.runner import AgentRunner
from agent.memory import MemoryStore
from agent.services import MockBackend, set_backend, get_backend
from agent.llm import using_real_llm
print('Gemini' if using_real_llm() else 'FakeLLM')
set_backend(MockBackend())
runner = AgentRunner(memory=MemoryStore(':memory:'), checkpointer=InMemorySaver())
emails = {e['id']: e for e in json.load(open('data/eval_dataset.json', encoding='utf-8'))}

## The graph

In [ ]:
print(runner.graph.get_graph().draw_mermaid())   # paste into https://mermaid.live to see it

## Milestone 1 — triage

In [ ]:
for eid in ['e01', 'e14', 'e23']:
    r = runner.process(emails[eid])
    print(eid, emails[eid]['subject'], '->', r['triage'], '|', r['status'])

## Milestone 3 — human-in-the-loop
The agent checks the calendar automatically (safe tool) and then **pauses** before `send_email`.

In [ ]:
res = runner.process(emails['e29'])
print(res['status'])
print(json.dumps(res['pending'], indent=2))
print(runner.trajectory(res['thread_id']))

In [ ]:
res = runner.resume(res['thread_id'], {'action': 'approve'})
print(res['status'], get_backend().sent[-1])

## Milestone 4 — memory (Bob → Robert)

In [ ]:
bob = {'id': 'nb1', 'from': 'bob.sharma@company.com', 'subject': 'Report', 'body': 'Can you send me the report by tomorrow?'}
r = runner.process(bob)
draft = r['pending']['args']['body']; print(draft)
r = runner.resume(r['thread_id'], {'action': 'edit', 'args': {'body': draft.replace('Bob', 'Robert')}, 'feedback': 'Call him Robert, not Bob. Use "Robert" instead of "Bob".'})
print('learned:', r['learned'])

In [ ]:
r2 = runner.process({**bob, 'id': 'nb2', 'subject': 'Notes', 'body': 'Could you share the retro notes?'})
print(r2['pending']['args']['body'])